# Poverty and Child Poverty Rates by State

This notebook calculates SPM poverty and child poverty rates for all 50 US states plus Washington DC using PolicyEngine microsimulation.

**Methodology:**
- Uses `person_in_poverty` variable (SPM poverty definition)
- Child poverty: persons under age 18 in poverty
- Weighted calculations using state-specific datasets

In [ ]:
from policyengine_us import Microsimulation
import pandas as pd
import numpy as np
from huggingface_hub import hf_hub_download
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
# Configuration
ANALYSIS_YEAR = 2026

# All 50 states + DC
STATES = [
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL", "GA",
    "HI", "ID", "IL", "IN", "IA", "KS", "KY", "LA", "ME", "MD",
    "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ",
    "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC",
    "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY",
    "DC"
]

# State names for display
STATE_NAMES = {
    "AL": "Alabama", "AK": "Alaska", "AZ": "Arizona", "AR": "Arkansas",
    "CA": "California", "CO": "Colorado", "CT": "Connecticut", "DE": "Delaware",
    "FL": "Florida", "GA": "Georgia", "HI": "Hawaii", "ID": "Idaho",
    "IL": "Illinois", "IN": "Indiana", "IA": "Iowa", "KS": "Kansas",
    "KY": "Kentucky", "LA": "Louisiana", "ME": "Maine", "MD": "Maryland",
    "MA": "Massachusetts", "MI": "Michigan", "MN": "Minnesota", "MS": "Mississippi",
    "MO": "Missouri", "MT": "Montana", "NE": "Nebraska", "NV": "Nevada",
    "NH": "New Hampshire", "NJ": "New Jersey", "NM": "New Mexico", "NY": "New York",
    "NC": "North Carolina", "ND": "North Dakota", "OH": "Ohio", "OK": "Oklahoma",
    "OR": "Oregon", "PA": "Pennsylvania", "RI": "Rhode Island", "SC": "South Carolina",
    "SD": "South Dakota", "TN": "Tennessee", "TX": "Texas", "UT": "Utah",
    "VT": "Vermont", "VA": "Virginia", "WA": "Washington", "WV": "West Virginia",
    "WI": "Wisconsin", "WY": "Wyoming", "DC": "District of Columbia"
}

In [ ]:
def get_state_dataset(state: str) -> str:
    """Download state-specific dataset from Hugging Face."""
    filename = f"states/{state}.h5"
    dataset_path = hf_hub_download(
        repo_id="policyengine/policyengine-us-data",
        filename=filename,
        repo_type="model",
    )
    return dataset_path


def calculate_poverty_rates(state: str, year: int = ANALYSIS_YEAR) -> dict:
    """
    Calculate poverty and child poverty rates for a state.
    
    Uses person_in_poverty variable (SPM poverty definition).
    Child poverty: age < 18.
    """
    # Get state dataset and run simulation
    dataset_path = get_state_dataset(state)
    sim = Microsimulation(dataset=dataset_path)
    
    # Get person-level poverty status (MicroSeries with weights)
    poverty = sim.calculate("person_in_poverty", year)
    age = sim.calculate("age", year)
    person_weight = sim.calculate("person_weight", year)
    
    # Overall poverty rate (weighted mean)
    poverty_rate = float(poverty.mean())
    
    # Child poverty rate (age < 18)
    child_mask = age < 18
    
    # For child poverty, we need to manually calculate weighted mean
    # since MicroSeries slicing may lose weights
    poverty_arr = np.array(poverty.values).astype(float)
    weight_arr = np.array(person_weight.values)
    age_arr = np.array(age.values)
    
    child_mask_arr = age_arr < 18
    
    if np.sum(weight_arr[child_mask_arr]) > 0:
        child_poverty_rate = float(
            np.average(poverty_arr[child_mask_arr], weights=weight_arr[child_mask_arr])
        )
    else:
        child_poverty_rate = 0.0
    
    # Population counts
    total_population = float(weight_arr.sum())
    child_population = float(weight_arr[child_mask_arr].sum())
    population_in_poverty = float((poverty_arr * weight_arr).sum())
    children_in_poverty = float((poverty_arr[child_mask_arr] * weight_arr[child_mask_arr]).sum())
    
    return {
        "state": state,
        "state_name": STATE_NAMES[state],
        "poverty_rate": poverty_rate,
        "child_poverty_rate": child_poverty_rate,
        "total_population": total_population,
        "child_population": child_population,
        "population_in_poverty": population_in_poverty,
        "children_in_poverty": children_in_poverty,
    }

In [ ]:
# Calculate poverty rates for all states
print(f"Calculating poverty rates for {len(STATES)} jurisdictions...\n")

results = []
for i, state in enumerate(STATES):
    print(f"[{i+1}/{len(STATES)}] Processing {STATE_NAMES[state]}...")
    try:
        state_results = calculate_poverty_rates(state, ANALYSIS_YEAR)
        results.append(state_results)
        print(f"    Poverty: {state_results['poverty_rate']:.1%}, Child poverty: {state_results['child_poverty_rate']:.1%}")
    except Exception as e:
        print(f"    ERROR: {e}")

print(f"\nCompleted {len(results)} states.")

In [ ]:
# Create DataFrame with results
df = pd.DataFrame(results)

# Format for display
df["poverty_rate_pct"] = df["poverty_rate"] * 100
df["child_poverty_rate_pct"] = df["child_poverty_rate"] * 100

# Sort by poverty rate (highest first)
df_sorted = df.sort_values("poverty_rate", ascending=False).reset_index(drop=True)

print(f"\n{'='*80}")
print(f"POVERTY RATES BY STATE ({ANALYSIS_YEAR})")
print(f"{'='*80}")
print(f"{'Rank':<6} {'State':<25} {'Poverty Rate':<15} {'Child Poverty Rate':<20}")
print("-" * 80)
for i, row in df_sorted.iterrows():
    print(f"{i+1:<6} {row['state_name']:<25} {row['poverty_rate_pct']:>10.1f}% {row['child_poverty_rate_pct']:>15.1f}%")

In [ ]:
# Summary statistics
print(f"\n{'='*60}")
print("SUMMARY STATISTICS")
print(f"{'='*60}")

print(f"\nOverall Poverty Rate:")
print(f"  Mean:   {df['poverty_rate_pct'].mean():.1f}%")
print(f"  Median: {df['poverty_rate_pct'].median():.1f}%")
print(f"  Min:    {df['poverty_rate_pct'].min():.1f}% ({df.loc[df['poverty_rate_pct'].idxmin(), 'state_name']})")
print(f"  Max:    {df['poverty_rate_pct'].max():.1f}% ({df.loc[df['poverty_rate_pct'].idxmax(), 'state_name']})")

print(f"\nChild Poverty Rate:")
print(f"  Mean:   {df['child_poverty_rate_pct'].mean():.1f}%")
print(f"  Median: {df['child_poverty_rate_pct'].median():.1f}%")
print(f"  Min:    {df['child_poverty_rate_pct'].min():.1f}% ({df.loc[df['child_poverty_rate_pct'].idxmin(), 'state_name']})")
print(f"  Max:    {df['child_poverty_rate_pct'].max():.1f}% ({df.loc[df['child_poverty_rate_pct'].idxmax(), 'state_name']})")

# Population-weighted national average
national_poverty = (df['population_in_poverty'].sum() / df['total_population'].sum()) * 100
national_child_poverty = (df['children_in_poverty'].sum() / df['child_population'].sum()) * 100

print(f"\nNational (Population-Weighted):")
print(f"  Poverty rate:       {national_poverty:.1f}%")
print(f"  Child poverty rate: {national_child_poverty:.1f}%")

In [ ]:
# Visualization: Bar chart of poverty rates
df_plot = df_sorted.head(20)  # Top 20 highest poverty

fig = go.Figure()

fig.add_trace(go.Bar(
    name="Overall Poverty",
    x=df_plot["state"],
    y=df_plot["poverty_rate_pct"],
    marker_color="#105293",
))

fig.add_trace(go.Bar(
    name="Child Poverty",
    x=df_plot["state"],
    y=df_plot["child_poverty_rate_pct"],
    marker_color="#F2994A",
))

fig.update_layout(
    title=f"Top 20 States by Poverty Rate ({ANALYSIS_YEAR})",
    xaxis_title="State",
    yaxis_title="Poverty Rate (%)",
    barmode="group",
    legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99),
)

fig.show()

In [ ]:
# Visualization: Choropleth map of poverty rates
fig_map = px.choropleth(
    df,
    locations="state",
    locationmode="USA-states",
    color="poverty_rate_pct",
    scope="usa",
    color_continuous_scale="Reds",
    labels={"poverty_rate_pct": "Poverty Rate (%)"},
    title=f"SPM Poverty Rate by State ({ANALYSIS_YEAR})",
    hover_name="state_name",
    hover_data={"poverty_rate_pct": ":.1f", "child_poverty_rate_pct": ":.1f", "state": False},
)

fig_map.update_layout(
    geo=dict(bgcolor="rgba(0,0,0,0)"),
)

fig_map.show()

In [ ]:
# Visualization: Child poverty choropleth
fig_child_map = px.choropleth(
    df,
    locations="state",
    locationmode="USA-states",
    color="child_poverty_rate_pct",
    scope="usa",
    color_continuous_scale="Oranges",
    labels={"child_poverty_rate_pct": "Child Poverty Rate (%)"},
    title=f"SPM Child Poverty Rate by State ({ANALYSIS_YEAR})",
    hover_name="state_name",
    hover_data={"poverty_rate_pct": ":.1f", "child_poverty_rate_pct": ":.1f", "state": False},
)

fig_child_map.update_layout(
    geo=dict(bgcolor="rgba(0,0,0,0)"),
)

fig_child_map.show()

In [ ]:
# Export to CSV
output_df = df[["state", "state_name", "poverty_rate_pct", "child_poverty_rate_pct", 
                "total_population", "child_population", "population_in_poverty", "children_in_poverty"]].copy()
output_df.columns = ["State Code", "State Name", "Poverty Rate (%)", "Child Poverty Rate (%)",
                     "Total Population", "Child Population", "Population in Poverty", "Children in Poverty"]
output_df = output_df.sort_values("Poverty Rate (%)", ascending=False)

output_df.to_csv("state_poverty_rates.csv", index=False)
print(f"Results saved to state_poverty_rates.csv")

In [ ]:
# Display full table
display_df = output_df.copy()
display_df["Poverty Rate (%)"] = display_df["Poverty Rate (%)"].round(1)
display_df["Child Poverty Rate (%)"] = display_df["Child Poverty Rate (%)"].round(1)
display_df["Total Population"] = display_df["Total Population"].apply(lambda x: f"{x:,.0f}")
display_df["Child Population"] = display_df["Child Population"].apply(lambda x: f"{x:,.0f}")
display_df["Population in Poverty"] = display_df["Population in Poverty"].apply(lambda x: f"{x:,.0f}")
display_df["Children in Poverty"] = display_df["Children in Poverty"].apply(lambda x: f"{x:,.0f}")
display_df